In [2]:
from prospect.models import priors, transforms #helper functions for specifying priors
from prospect.models import sedmodel
def build_model(**kwargs):
    
    """
    Function to build model components for SFH and dust. 
    The model params are defined by their name, whether they are a free parameter
    their initial value, and their prior distribution if they are variable. The model 
    params are then fed to the prospector SedModel class
    
    All parameters except 'mass' correspond to fsps model parameters, the definitions of which you can find here:
    https://dfm.io/python-fsps/current/stellarpop_api/
    
    """
    
    model_params = []
    #luminosity distance of galaxy. for a z=0 simba galaxy, i typically just set this to be 10 pc
    model_params.append({'name': "lumdist", "N": 1, "isfree": False,"init": 1.0e-5,"units": "Mpc"})
    #IMF model which will be used by the simple stellar population model
    model_params.append({'name': 'imf_type', 'N': 1,'isfree': False,'init': 2, 'prior': None})
    #stellar mass of a galaxy -- what we're interested in! So we'll set it as a free parameter
    model_params.append({'name': 'mass', 'N': 1,'isfree': True, 'init': 1e10,'prior': priors.TopHat(mini=1e8, maxi=1e12)})
    #stellar metallicity, in units of log(Z/Z_sun)
    model_params.append({'name': 'logzsol', 'N': 1,'isfree': True,'init': -0.5,'prior': priors.TopHat(mini=-1.6, maxi=0.1)})
    #SFH model. here, we are choosing the 'delayed-tau' model and has two free parameters: the age and the e-folding time
    model_params.append({'name': "sfh", "N": 1, "isfree": False, "init": 4, 'prior': None})
    #age of the galaxy
    model_params.append({'name': "tage", 'N': 1, 'isfree': True, 'init': 5., 'units': 'Gyr', 'prior': priors.TopHat(mini=0.001, maxi=13.8)})
    #e-folding time
    model_params.append({'name': "tau", 'N': 1, 'isfree': True,'init': 1., 'units': 'Gyr', 'prior': priors.LogUniform(mini=0.1, maxi=30)})
    #dust attenuation model, from Calzetti 2001
    model_params.append({'name': 'dust_type', 'N': 1,'isfree': False,'init': 2,'prior': None})
    #the attenuation (in magnitudes) in the V-band
    model_params.append({'name': 'dust2', 'N': 1,'isfree': True, 'init': 0.1,'prior': priors.ClippedNormal(mini=0.0, maxi=2.0, mean=0.0, sigma=0.3)})
    #dust emission model -- only 1 choice, from Draine & Li 2007
    model_params.append({'name': 'add_dust_emission', 'N': 1,'isfree': False,'init': 1,'prior': None})
    #mass fraction of warm dust
    model_params.append({'name': 'duste_gamma', 'N': 1,'isfree': True,'init': 0.01,'prior': priors.TopHat(mini=0.0, maxi=1.0)})
    #minimum radiation field
    model_params.append({'name': 'duste_umin', 'N': 1,'isfree': True,'init': 1.0,'prior': priors.TopHat(mini=0.1, maxi=20.0)})
    #mass fraction of dust in PAHs
    model_params.append({'name': 'duste_qpah', 'N': 1,'isfree': False,'init': 3.0,'prior': priors.TopHat(mini=0.0, maxi=6.0)})
    
    
    model = sedmodel.SedModel(model_params)
    return model

In [3]:
mod = build_model()
print(mod)

:::::::
<class 'prospect.models.sedmodel.SedModel'>

Free Parameters: (name: prior) 
-----------
  mass: <class 'prospect.models.priors.TopHat'>(mini=100000000.0,maxi=1000000000000.0)
  logzsol: <class 'prospect.models.priors.TopHat'>(mini=-1.6,maxi=0.1)
  tage: <class 'prospect.models.priors.TopHat'>(mini=0.001,maxi=13.8)
  tau: <class 'prospect.models.priors.LogUniform'>(mini=0.1,maxi=30)
  dust2: <class 'prospect.models.priors.ClippedNormal'>(mean=0.0,sigma=0.3,mini=0.0,maxi=2.0)
  duste_gamma: <class 'prospect.models.priors.TopHat'>(mini=0.0,maxi=1.0)
  duste_umin: <class 'prospect.models.priors.TopHat'>(mini=0.1,maxi=20.0)

Fixed Parameters: (name: value [, depends_on]) 
-----------
  lumdist: [1.e-05] 
  imf_type: [2] 
  sfh: [4] 
  dust_type: [2] 
  add_dust_emission: [1] 
  duste_qpah: [3.] 


In [5]:
from prospect.sources import CSPSpecBasis
def build_sps(**kwargs):
    """
    This is our stellar population model which generates the spectra for stars of a given age and mass. 
    Most of the time, you aren't going to need to pay attention to this. 
    """
    sps = CSPSpecBasis(zcontinuous=1)
    return sps

In [6]:
sps = build_sps()